In [1]:
import pandas as pd

from sklearn.utils import resample
from sklearn.preprocessing import MinMaxScaler

In [2]:
train = pd.read_csv("train_pre.csv")
test = pd.read_csv("test_pre.csv")

In [3]:
# 小数点第2位以下を丸め
train['緯度'] = train['緯度'].round(2)
train['経度'] = train['経度'].round(2)
test['緯度'] = test['緯度'].round(2)
test['経度'] = test['経度'].round(2)

In [4]:
# 正規化したい列（IDと賃料を除く）
columns_to_normalize = train.columns.difference(['賃料'])

In [5]:
# MinMaxScalerのインスタンスを作成
scaler = MinMaxScaler()

In [6]:
# 指定した列を正規化
train[columns_to_normalize] = scaler.fit_transform(train[columns_to_normalize])
test[columns_to_normalize] = scaler.fit_transform(test[columns_to_normalize])

# オーバーサンプリングを行うための関数
def oversample(df, target_column, threshold):
    # 各クラスのカウント
    counts = df[target_column].value_counts()
    
    # オーバーサンプリングが必要なクラスを特定
    to_oversample = counts[counts < threshold].index
    
    # オーバーサンプリングを実施
    for cls in to_oversample:
        n_samples = threshold - counts[cls]
        cls_samples = df[df[target_column] == cls]
        oversampled_cls = resample(cls_samples, 
                                    replace=True, 
                                    n_samples=n_samples, 
                                    random_state=42)
        df = pd.concat([df, oversampled_cls], ignore_index=True)
    
    return df

In [7]:
# オーバーサンプリングとアンダーサンプリングを行う関数
def oversample_undersample(df, target_column, threshold = 2500):
    # 各クラスのカウント
    counts = df[target_column].value_counts()
    
    # オーバーサンプリングが必要なクラスを特定
    to_oversample = counts[counts < threshold].index
    # アンダーサンプリングが必要なクラスを特定
    to_undersample = counts[counts > threshold].index

    # オーバーサンプリングを実施
    for cls in to_oversample:
        n_samples = threshold - counts[cls]
        cls_samples = df[df[target_column] == cls]
        oversampled_cls = resample(cls_samples, 
                                    replace=True, 
                                    n_samples=n_samples, 
                                    random_state=42)
        df = pd.concat([df, oversampled_cls], ignore_index=True)

    # アンダーサンプリングを実施
    for cls in to_undersample:
        n_samples = threshold
        cls_samples = df[df[target_column] == cls]
        undersampled_cls = resample(cls_samples, 
                                    replace=False, 
                                    n_samples=n_samples, 
                                    random_state=42)
        df = df[df[target_column] != cls]  # 元のクラスを削除
        df = pd.concat([df, undersampled_cls], ignore_index=True)
    
    return df

In [9]:
# リサンプリングの実行
#balanced_train = oversample_undersample(train, 'アクセス')
balanced_train = oversample_undersample(train, '築年数')
balanced_train = oversample_undersample(balanced_train, '面積')
balanced_train = oversample_undersample(balanced_train, '建物構造')
#balanced_train = oversample_undersample(balanced_train, '最上階フラグ')
balanced_train = oversample_undersample(balanced_train, '間取り')
balanced_train = oversample_undersample(balanced_train, '緯度')
balanced_train = oversample_undersample(balanced_train, '経度')

In [10]:
balanced_train.to_csv('train_balanced_pre.csv', index=False)
test.to_csv('test_pre2.csv', index=False)